In [1]:
from scheduler.service.cv_ai_trainer_s1_gen02 import *

2025-05-29 16:39:46.250452: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-29 16:39:46.250618: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [17]:
cv_ai_trainer = CvAiTrainerS1Gen02(seq=11)

In [18]:
df = cv_ai_trainer.get_dataframe()

In [19]:
# define features
features = ['timestamp', 'weekday', 'open', 'high', 'low', 'close', 'volume', 'volume_ma5', 'volume_ma20', 'volume_ratio', 'obv', 'vwap', 'volume_volatility', 'volume_spike', 'rsi', 'macd', 'macd_signal', 'bollinger_partition', 'bollinger_partition_prev', 'atr', 'cci', 'stochastic_k', 'stochastic_d', 'roc', 'momentum', 'ema_cross_up', 'cdl2crows', 'cdl3blackcrows', 'cdl3inside', 'cdl3linestrike', 'cdl3outside', 'cdl3starsinsouth', 'cdl3whitesoldiers', 'cdlabandonedbaby', 'cdladvanceblock', 'cdlbelthold', 'cdlbreakaway', 'cdlclosingmarubozu', 'cdlconcealbabyswall', 'cdlcounterattack', 'cdldarkcloudcover', 'cdldoji', 'cdldojistar', 'cdldragonflydoji', 'cdlengulfing', 'cdleveningdojistar', 'cdleveningstar', 'cdlgapsidesidewhite', 'cdlgravestonedoji', 'cdlhammer', 'cdlhangingman', 'cdlharami', 'cdlharamicross', 'cdlhighwave', 'cdlhikkake', 'cdlhikkakemod', 'cdlhomingpigeon', 'cdlidentical3crows', 'cdlinneck', 'cdlinvertedhammer', 'cdlkicking', 'cdlkickingbylength', 'cdlladderbottom', 'cdllongleggeddoji', 'cdllongline', 'cdlmarubozu', 'cdlshortline', 'cdlspinningtop', 'cdlstalledpattern', 'cdlsticksandwich', 'cdltakuri', 'cdltasukigap', 'cdlthrusting', 'cdltristar', 'cdlunique3river', 'cdlupsidegap2crows', 'cdlxsidegap3methods', 'high_low_ratio', 'close_open_ratio', 'daily_range', 'price_change_ratio', 'high_low_ratio', 'rolling_volatility', 'close_above_ma5', 'candle_body_size', 'wick_ratio', 'volatility_1h', 'gap_vs_30min_avg', 'bollinger_width', 'close_vs_ma12', 'close_vs_ma36']
print(features)

['timestamp', 'weekday', 'open', 'high', 'low', 'close', 'volume', 'volume_ma5', 'volume_ma20', 'volume_ratio', 'obv', 'vwap', 'volume_volatility', 'volume_spike', 'rsi', 'macd', 'macd_signal', 'bollinger_partition', 'bollinger_partition_prev', 'atr', 'cci', 'stochastic_k', 'stochastic_d', 'roc', 'momentum', 'ema_cross_up', 'cdl2crows', 'cdl3blackcrows', 'cdl3inside', 'cdl3linestrike', 'cdl3outside', 'cdl3starsinsouth', 'cdl3whitesoldiers', 'cdlabandonedbaby', 'cdladvanceblock', 'cdlbelthold', 'cdlbreakaway', 'cdlclosingmarubozu', 'cdlconcealbabyswall', 'cdlcounterattack', 'cdldarkcloudcover', 'cdldoji', 'cdldojistar', 'cdldragonflydoji', 'cdlengulfing', 'cdleveningdojistar', 'cdleveningstar', 'cdlgapsidesidewhite', 'cdlgravestonedoji', 'cdlhammer', 'cdlhangingman', 'cdlharami', 'cdlharamicross', 'cdlhighwave', 'cdlhikkake', 'cdlhikkakemod', 'cdlhomingpigeon', 'cdlidentical3crows', 'cdlinneck', 'cdlinvertedhammer', 'cdlkicking', 'cdlkickingbylength', 'cdlladderbottom', 'cdllongleggeddo

In [20]:
# define targets
targets = ['change_after_1day']
print(targets)

['change_after_1day']


In [22]:
# define preprocess
from sklearn.preprocessing import StandardScaler
import numpy as np

# 전처리 - 수치화
df = df.apply(pd.to_numeric, errors='coerce')

# 파생변수 -
df['high_low_ratio'] = df['high'] / df['low']
df['close_open_ratio'] = df['close'] / df['open']
df['daily_range'] = (df['high'] - df['low']) / df['close']
df['price_change_ratio'] = (df['close'] - df['open']) / df['open']
df['rolling_volatility'] = df['close'].pct_change().rolling(10).std()
df['close_above_ma5'] = (df['close'] > df['close'].rolling(5).mean()).astype(int)

# 파생변수 - 고저 비율 (급등락 감지)
df['high_low_ratio'] = df['high'] / df['low']

# 파생변수 - 캔들 몸통 크기
df['candle_body_size'] = (df['close'] - df['open']).abs()

# 파생변수 - 캔들 윗꼬리 & 아랫꼬리 비율
df['upper_wick'] = df['high'] - df[['open', 'close']].max(axis=1)
df['lower_wick'] = df[['open', 'close']].min(axis=1) - df['low']
df['wick_ratio'] = (df['upper_wick'] + df['lower_wick']) / df['high'] - df['low'] + 1e-6

# 파생변수 - 1시간(=12개) 변동성 지표
df['volatility_1h'] = df['close'].pct_change().rolling(12).std()

# 파생변수 - 30분 평균 대비 갭
df['gap_vs_30min_avg'] = (df['open'] - df['close'].shift(1)) / df['close'].shift(1).rolling(6).mean()

# 파생변수 - 볼린저 밴드 너비 (20개 = 100분)
df['bollinger_width'] = (
    4 * df['close'].rolling(20).std() / df['close'].rolling(20).mean()
)

# 파생변수 - 이동평균 대비 종가
df['close_vs_ma12'] = df['close'] / df['close'].rolling(12).mean()  # 1시간 평균
df['close_vs_ma36'] = df['close'] / df['close'].rolling(36).mean()  # 3시간 평균

# 동적 타겟 생성
change_pct = (df['price_after_1day'] - df['price_on_time']) / df['price_on_time'] * 100
df['change_after_1day'] = np.select(
    [
        change_pct > 1,
        change_pct < -1
    ],
    [
        2,  # 상승
        0   # 하락
    ],
    default=1  # 횡보
)

# 전처리 - 결측치 제외
df.fillna(df.mean(numeric_only=True), inplace=True)

# 전처리 - 스케일링
df[features] = StandardScaler().fit_transform(df[features])

print(df.columns.tolist())

['seq', 'ticker', 'tick_datetime', 'timestamp', 'weekday', 'open', 'high', 'low', 'close', 'volume', 'volume_ma5', 'volume_ma20', 'volume_ratio', 'obv', 'vwap', 'volume_volatility', 'volume_spike', 'rsi', 'macd', 'macd_signal', 'bollinger_partition', 'bollinger_partition_prev', 'atr', 'cci', 'stochastic_k', 'stochastic_d', 'roc', 'momentum', 'ema_cross_up', 'cdl2crows', 'cdl3blackcrows', 'cdl3inside', 'cdl3linestrike', 'cdl3outside', 'cdl3starsinsouth', 'cdl3whitesoldiers', 'cdlabandonedbaby', 'cdladvanceblock', 'cdlbelthold', 'cdlbreakaway', 'cdlclosingmarubozu', 'cdlconcealbabyswall', 'cdlcounterattack', 'cdldarkcloudcover', 'cdldoji', 'cdldojistar', 'cdldragonflydoji', 'cdlengulfing', 'cdleveningdojistar', 'cdleveningstar', 'cdlgapsidesidewhite', 'cdlgravestonedoji', 'cdlhammer', 'cdlhangingman', 'cdlharami', 'cdlharamicross', 'cdlhighwave', 'cdlhikkake', 'cdlhikkakemod', 'cdlhomingpigeon', 'cdlidentical3crows', 'cdlinneck', 'cdlinvertedhammer', 'cdlkicking', 'cdlkickingbylength', '

In [23]:
# define hyperparameters
hyperparameters = {}
hyperparameters["time_steps"] = 40
hyperparameters["batch_size"] = 32
hyperparameters["epochs"] = 20

In [24]:
# define model training
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# 피쳐, 타겟 분리
X = df[features]
y = df['change_after_1day']

# 드롭아웃
X = X.select_dtypes(exclude=['datetime'])

# 학습/검증 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 훈련
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [26]:
# define model test
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 학습 데이터 정확도
y_train_pred = model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
print(f"✅ 학습 정확도: {train_acc:.4f}")

# 테스트 데이터 정확도
y_test_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)
print(f"✅ 테스트 정확도: {test_acc:.4f}")

# 상세 리포트
print("\n📋 분류 리포트 (테스트 데이터):")
print(classification_report(y_test, y_test_pred, target_names=['하락', '횡보', '상승']))

✅ 학습 정확도: 1.0000
✅ 테스트 정확도: 0.9458

📋 분류 리포트 (테스트 데이터):
              precision    recall  f1-score   support

          하락       0.95      0.92      0.93       965
          횡보       0.93      0.96      0.95      2325
          상승       0.96      0.95      0.95      1453

    accuracy                           0.95      4743
   macro avg       0.95      0.94      0.94      4743
weighted avg       0.95      0.95      0.95      4743

